# Agente de IA Minimalista - v2: análise temporal (Gemini)

**O que esta versão acrescenta ao notebook do professor:**

1. **Nova tool `analisar_serie_temporal`**: recebe um serviço e uma métrica e devolve *quando* a métrica mudou de
   comportamento (ponto de mudança), a taxa de crescimento depois disso e quais eventos/deploys aconteceram
   perto desse momento (antes ou depois).
2. **Novo problema (cenário B, INC-2026-0912):** vazamento de memória gradual no `catalog-api`, cujo sintoma só aparece
   horas depois no `checkout-api`, com um deploy recente e inocente no checkout como "pista falsa".
3. **Séries temporais no mockup** (geradas de forma determinística, sempre os mesmos números).
4. **Testes e avaliação:** o agente roda várias vezes em cada cenário, com e sem a tool temporal, e a tabela final mostra
   acerto, rodadas, tokens e custo.
5. **Portado para Gemini** (API nativa, sem bibliotecas extras) e com nova tentativa automática em erros 429/503.
6. **Instruções melhoradas** a partir de falhas observadas no teste do cenário original (veja o relatório).

In [1]:
import sys
print("Kernel OK. Python:", sys.version.split()[0])

Kernel OK. Python: 3.11.9


In [2]:
import json
import math
import os
import random
import time
import urllib.error
import urllib.request
from copy import deepcopy
from getpass import getpass
from pathlib import Path

ARQUIVO_CHAVE = Path("gemini_key.txt")   # arquivo de texto com a chave, na mesma pasta do notebook

if not os.environ.get("GEMINI_API_KEY"):
    if ARQUIVO_CHAVE.exists():
        os.environ["GEMINI_API_KEY"] = ARQUIVO_CHAVE.read_text(encoding="utf-8").strip()
        print("Chave lida do arquivo gemini_key.txt")
    else:
        os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY: ").strip()

if not os.environ["GEMINI_API_KEY"]:
    raise RuntimeError("Chave vazia. Coloque a chave no arquivo gemini_key.txt.")
print(f"Chave carregada ({len(os.environ['GEMINI_API_KEY'])} caracteres).")

BASE_URL = "https://generativelanguage.googleapis.com/v1beta"
MODEL = "gemini-3.1-flash-lite"
MAX_RODADAS = 8
MAX_OUTPUT_TOKENS = 1200
THINKING_LEVEL = "low"

# Plano gratuito: custo real zero. Precos de referencia (GPT-5.6 Luna, do notebook original)
# usados so para calcular o "custo equivalente".
LIMITE_ESTIMADO_USD = 0.02
PRECO_INPUT_1M = 0.20
PRECO_OUTPUT_1M = 1.20

Chave lida do arquivo gemini_key.txt
Chave carregada (53 caracteres).


## Mockup: utilitários de tempo e gerador de séries temporais

Cada série é gerada com `random.Random(semente)`; portanto **os números são sempre os mesmos** em qualquer computador.

In [3]:
SERVICOS = ["api-gateway", "recommendation-api", "catalog-api", "checkout-api"]
METRICAS = ["cpu_pct", "memoria_pct", "latencia_p95_ms", "taxa_erro_pct"]
SD_RUIDO = {"cpu_pct": 1.2, "memoria_pct": 0.5, "taxa_erro_pct": 0.05}   # latencia: 3% do valor


def hhmm_para_min(h):
    hh, mm = h.split(":")
    return int(hh) * 60 + int(mm)


def min_para_hhmm(m):
    m = int(round(m))
    return f"{m // 60:02d}:{m % 60:02d}"


def rampa(t, t0, duracao, delta):
    return delta * min(1.0, max(0.0, (t - t0) / duracao))


def gerar_series(inicio, fim, passo, funcao_valor, semente):
    rng = random.Random(semente)
    tempos = range(hhmm_para_min(inicio), hhmm_para_min(fim) + 1, passo)
    dados = {s: {m: [] for m in METRICAS} for s in SERVICOS}
    for t in tempos:
        for s in SERVICOS:
            for m in METRICAS:
                base = funcao_valor(s, m, t)
                sd = 0.03 * base if m == "latencia_p95_ms" else SD_RUIDO[m]
                dados[s][m].append(round(max(0.0, base + rng.gauss(0, sd)), 2))
    return {"inicio": inicio, "fim": fim, "passo_min": passo, "dados": dados}

## Cenário A: o original do professor (incidente INC-2026-0911)

Mantive os dados originais e apenas acrescentei uma série temporal coerente com eles (para a nova tool funcionar aqui também).

In [4]:
METRICAS_ATUAIS_A = {
    "api-gateway": {"cpu_pct": 38, "memoria_pct": 61, "latencia_p95_ms": 6200, "taxa_erro_pct": 18.4, "instancias": 3},
    "recommendation-api": {"cpu_pct": 98, "memoria_pct": 84, "latencia_p95_ms": 5870, "taxa_erro_pct": 31.7, "instancias": 48},
    "catalog-api": {"cpu_pct": 42, "memoria_pct": 55, "latencia_p95_ms": 93, "taxa_erro_pct": 0.2, "instancias": 4},
    "checkout-api": {"cpu_pct": 47, "memoria_pct": 58, "latencia_p95_ms": 310, "taxa_erro_pct": 0.6, "instancias": 5},
}


def valor_A(s, m, t):
    h = hhmm_para_min
    atual = METRICAS_ATUAIS_A[s][m]
    if s == "recommendation-api":
        antes = {"cpu_pct": 35, "memoria_pct": 52, "latencia_p95_ms": 180, "taxa_erro_pct": 0.4}[m]
        if m in ("cpu_pct", "memoria_pct"):
            return antes + rampa(t, h("13:58"), 5, atual - antes)
        return antes + rampa(t, h("14:00"), 3, atual - antes)
    if s == "api-gateway" and m in ("latencia_p95_ms", "taxa_erro_pct"):
        antes = 140 if m == "latencia_p95_ms" else 0.3
        return antes + rampa(t, h("14:03"), 2, atual - antes)
    return atual


SERIES_A = gerar_series("13:00", "14:08", 1, valor_A, semente=911)
for s in SERVICOS:                       # o ultimo ponto da serie e o valor atual do snapshot
    for m in METRICAS:
        SERIES_A["dados"][s][m][-1] = METRICAS_ATUAIS_A[s][m]

CENARIO_A = {
    "id": "A_original",
    "incidente": (
        "INC-2026-0911: desde 14h03, usuarios relatam lentidao no e-commerce. "
        "O api-gateway disparou alerta de latencia e o custo cloud esta subindo. "
        "Descubra a causa raiz e recomende a resposta operacional mais segura."
    ),
    "agora": "14:08",
    "metricas": METRICAS_ATUAIS_A,
    "logs": {
        "api-gateway": [
            "14:03:11 WARN upstream recommendation-api timeout after 5000ms",
            "14:03:14 WARN upstream recommendation-api timeout after 5000ms",
            "14:03:18 ERROR circuit breaker recommendation-api OPEN",
        ],
        "recommendation-api": [
            "14:02:57 WARN catalog response missing field related_items",
            "14:02:57 INFO invoking fallback target=recommendation-api depth=1",
            "14:02:58 INFO invoking fallback target=recommendation-api depth=8",
            "14:02:59 ERROR maximum fallback depth reached depth=25",
            "14:03:00 WARN request queue saturation=96%",
        ],
        "catalog-api": [
            "14:02:55 INFO GET /items/842 200 71ms",
            "14:03:01 INFO GET /items/163 200 66ms",
        ],
        "checkout-api": [
            "14:03:02 INFO POST /checkout 200 288ms",
            "14:03:07 INFO POST /checkout 200 302ms",
        ],
    },
    "deploys": {
        "api-gateway": [{"id": "dep-410", "versao": "5.8.1", "hora": "ontem 16:20", "status": "concluido"}],
        "recommendation-api": [
            {"id": "dep-442", "versao": "2.4.0", "hora": "13:58", "status": "concluido",
             "mudancas": ["novo fallback para resposta incompleta do catalogo",
                          "FALLBACK_SERVICE: catalog-api -> recommendation-api"]},
            {"id": "dep-431", "versao": "2.3.7", "hora": "ontem 11:40", "status": "concluido"},
        ],
        "catalog-api": [{"id": "dep-405", "versao": "3.1.2", "hora": "3 dias atras", "status": "concluido"}],
        "checkout-api": [{"id": "dep-398", "versao": "7.2.0", "hora": "5 dias atras", "status": "concluido"}],
    },
    "configuracoes": {
        "api-gateway": {"timeout_ms": 5000, "retries": 1},
        "recommendation-api": {"fallback_service": "recommendation-api", "fallback_max_depth": 25,
                               "autoscaling_min": 2, "autoscaling_max": 50, "autoscaling_cpu_target_pct": 60},
        "catalog-api": {"cache_ttl_s": 300, "replicas": 4},
        "checkout-api": {"replicas": 5, "timeout_ms": 1500},
    },
    "custos": {
        "api-gateway": {"baseline_usd_h": 0.22, "atual_usd_h": 0.24},
        "recommendation-api": {"baseline_usd_h": 0.31, "atual_usd_h": 7.44},
        "catalog-api": {"baseline_usd_h": 0.38, "atual_usd_h": 0.40},
        "checkout-api": {"baseline_usd_h": 0.51, "atual_usd_h": 0.52},
    },
    "linha_do_tempo": [
        {"hora": "13:58", "tipo": "deploy", "servico": "recommendation-api", "id": "dep-442"},
        {"hora": "14:01", "tipo": "autoscaling", "servico": "recommendation-api", "detalhe": "2 -> 18 instancias"},
        {"hora": "14:03", "tipo": "autoscaling", "servico": "recommendation-api", "detalhe": "18 -> 48 instancias"},
        {"hora": "14:03", "tipo": "alerta", "servico": "api-gateway", "detalhe": "latencia p95 acima de 5s"},
    ],
    "series": SERIES_A,
}
print("Cenario A pronto:", len(SERIES_A["dados"]["api-gateway"]["cpu_pct"]), "pontos por serie (1 min).")

Cenario A pronto: 69 pontos por serie (1 min).


## Cenário B (NOVO): vazamento de memória gradual (incidente INC-2026-0912)

**História (a que o agente precisa reconstruir):**
- 12:10: deploy `dep-451` no `catalog-api` troca o TTL do cache de 300 s para 0 (o cache nunca expira). A memória do catalog começa a crescer devagar.
- ~15:30: o heap chega a ~85% e começam pausas longas de coleta de lixo (GC); a latência do catalog dispara.
- ~15:39: o `checkout-api`, que depende do catalog, começa a ficar lento.
- 15:52: deploy `dep-455` no `checkout-api` (só um ajuste visual de banner). **É a pista falsa: ele aparece DEPOIS do início do sintoma.**
- 16:05: autoscaling do checkout (5 -> 8). 16:10: alerta de latência.

Sem olhar o tempo, o "último deploy" (dep-455) parece o culpado. A tool temporal mostra que o checkout já estava piorando antes dele
e que a memória do catalog mudou de tendência exatamente às 12:10.

In [5]:
def valor_B(s, m, t):
    h = hhmm_para_min
    t_deploy, t_gc = h("12:10"), h("15:30")
    mem_cat = 55 + (0.15 * (t - t_deploy) if t > t_deploy else 0)
    gc = max(0, t - t_gc)
    lat_cat = 90 + 12 * gc
    lat_chk = 310 + 1.8 * max(0, lat_cat - 200)
    if s == "catalog-api":
        return {"cpu_pct": 42 + 0.4 * gc, "memoria_pct": mem_cat,
                "latencia_p95_ms": lat_cat, "taxa_erro_pct": 0.2 + 0.05 * gc}[m]
    if s == "checkout-api":
        return {"cpu_pct": 47, "memoria_pct": 58, "latencia_p95_ms": lat_chk,
                "taxa_erro_pct": 0.6 + 0.12 * max(0, gc - 9)}[m]
    if s == "api-gateway":
        return {"cpu_pct": 38, "memoria_pct": 61,
                "latencia_p95_ms": 120 + 0.3 * max(0, lat_chk - 310), "taxa_erro_pct": 0.3}[m]
    return {"cpu_pct": 44, "memoria_pct": 57, "latencia_p95_ms": 150, "taxa_erro_pct": 0.4}[m]


SERIES_B = gerar_series("11:00", "16:15", 5, valor_B, semente=912)
INSTANCIAS_B = {"api-gateway": 3, "recommendation-api": 2, "catalog-api": 4, "checkout-api": 8}
METRICAS_ATUAIS_B = {
    s: {**{m: SERIES_B["dados"][s][m][-1] for m in METRICAS}, "instancias": INSTANCIAS_B[s]} for s in SERVICOS
}

CENARIO_B = {
    "id": "B_vazamento_de_memoria",
    "incidente": (
        "INC-2026-0912: desde cerca de 16h10, o checkout esta lento e o alerta de latencia p95 do checkout-api disparou. "
        "Clientes relatam demora para finalizar pedidos. "
        "Descubra a causa raiz e recomende a resposta operacional mais segura."
    ),
    "agora": "16:15",
    "metricas": METRICAS_ATUAIS_B,
    "logs": {
        "api-gateway": [
            "16:04 WARN upstream checkout-api slow response 1.4s",
            "16:11 WARN upstream checkout-api slow response 1.9s",
        ],
        "recommendation-api": ["16:10 INFO GET /recommendations 200 148ms"],
        "catalog-api": [
            "15:33 WARN GC pause 380ms heap=86%",
            "15:41 WARN GC pause 640ms heap=87%",
            "16:07 ERROR GC pause 1900ms heap=91%",
        ],
        "checkout-api": [
            "15:44 WARN call catalog-api /items latency 480ms",
            "16:03 WARN timeout calling catalog-api after 800ms retry=1",
            "16:09 ERROR POST /checkout 504 after 2100ms",
        ],
    },
    "deploys": {
        "api-gateway": [{"id": "dep-410", "versao": "5.8.1", "hora": "ontem 16:20", "status": "concluido"}],
        "recommendation-api": [{"id": "dep-431", "versao": "2.3.7", "hora": "ontem 11:40", "status": "concluido"}],
        "catalog-api": [
            {"id": "dep-451", "versao": "3.2.0", "hora": "12:10", "status": "concluido",
             "mudancas": ["cache de itens em memoria para acelerar listagens", "cache_ttl_s: 300 -> 0 (sem expiracao)"]},
            {"id": "dep-405", "versao": "3.1.2", "hora": "3 dias atras", "status": "concluido"},
        ],
        "checkout-api": [
            {"id": "dep-455", "versao": "7.3.0", "hora": "15:52", "status": "concluido",
             "mudancas": ["ajuste de texto e cor do banner promocional"]},
            {"id": "dep-398", "versao": "7.2.0", "hora": "5 dias atras", "status": "concluido"},
        ],
    },
    "configuracoes": {
        "api-gateway": {"timeout_ms": 5000, "retries": 1},
        "recommendation-api": {"autoscaling_min": 2, "autoscaling_max": 50},
        "catalog-api": {"cache_ttl_s": 0, "cache_max_entries": "ilimitado", "heap_max_mb": 4096, "replicas": 4},
        "checkout-api": {"replicas": 8, "timeout_ms": 1500, "catalog_timeout_ms": 800},
    },
    "custos": {
        "api-gateway": {"baseline_usd_h": 0.22, "atual_usd_h": 0.22},
        "recommendation-api": {"baseline_usd_h": 0.31, "atual_usd_h": 0.31},
        "catalog-api": {"baseline_usd_h": 0.38, "atual_usd_h": 0.38},
        "checkout-api": {"baseline_usd_h": 0.51, "atual_usd_h": 0.82},
    },
    "linha_do_tempo": [
        {"hora": "12:10", "tipo": "deploy", "servico": "catalog-api", "id": "dep-451"},
        {"hora": "15:52", "tipo": "deploy", "servico": "checkout-api", "id": "dep-455"},
        {"hora": "16:05", "tipo": "autoscaling", "servico": "checkout-api", "detalhe": "5 -> 8 instancias"},
        {"hora": "16:10", "tipo": "alerta", "servico": "checkout-api", "detalhe": "latencia p95 acima de 900ms"},
    ],
    "series": SERIES_B,
}

CENARIOS = {"A": CENARIO_A, "B": CENARIO_B}
ambiente = deepcopy(CENARIO_B)


def carregar_cenario(nome):
    global ambiente
    ambiente = deepcopy(CENARIOS[nome])
    return ambiente["id"]


print("Cenario B pronto:", len(SERIES_B["dados"]["catalog-api"]["memoria_pct"]), "pontos por serie (5 min).")
print("Snapshot atual do checkout-api:", METRICAS_ATUAIS_B["checkout-api"])

Cenario B pronto: 64 pontos por serie (5 min).
Snapshot atual do checkout-api: {'cpu_pct': 47.98, 'memoria_pct': 58.88, 'latencia_p95_ms': 1068.99, 'taxa_erro_pct': 4.86, 'instancias': 8}


## Ferramentas

As 7 ferramentas originais continuam iguais, exceto `consultar_eventos_recentes`, que agora só mostra os eventos dos **últimos 90 minutos**
(como um painel real faria). Isso esconde o deploy das 12:10 e deixa em destaque o das 15:52.
A ferramenta nova é `analisar_serie_temporal`.

In [6]:
def _validar_servico(servico):
    if servico not in ambiente["metricas"]:
        return {"erro": f"Servico desconhecido: {servico}", "disponiveis": list(ambiente["metricas"])}
    return None


def consultar_metricas(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, **ambiente["metricas"][servico]}


def buscar_logs(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "linhas": ambiente["logs"].get(servico, [])}


def consultar_deploys(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "deploys": ambiente["deploys"].get(servico, [])}


def consultar_configuracao(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "configuracao": ambiente["configuracoes"].get(servico, {})}


def consultar_custos(servico):
    erro = _validar_servico(servico)
    if erro:
        return erro
    custo = ambiente["custos"][servico]
    return {"servico": servico, **custo, "multiplicador": round(custo["atual_usd_h"] / custo["baseline_usd_h"], 1)}


def consultar_eventos_recentes():
    agora = hhmm_para_min(ambiente["agora"])
    recentes = [e for e in ambiente["linha_do_tempo"] if hhmm_para_min(e["hora"]) >= agora - 90]
    return {"agora": ambiente["agora"], "janela_min": 90, "eventos": recentes}


def solicitar_rollback(deploy_id, justificativa):
    # Esta ferramenta NAO executa o rollback. Ela abre uma solicitacao.
    encontrado = None
    for servico, deploys in ambiente["deploys"].items():
        for deploy in deploys:
            if deploy["id"] == deploy_id:
                encontrado = {"servico": servico, **deploy}
                break
    if not encontrado:
        return {"status": "rejeitado", "motivo": "deploy inexistente"}
    return {"status": "AGUARDANDO_APROVACAO_HUMANA", "deploy": encontrado,
            "justificativa_registrada": justificativa, "executado": False}

### A ferramenta nova: `analisar_serie_temporal`

**Como funciona (em português simples):**
1. Pega a série da métrica (ex.: memória do catalog-api de 11:00 a 16:15).
2. Testa cada horário possível como "o momento em que a métrica mudou": antes dele a série é uma linha plana, depois dele é uma
   reta inclinada. O horário em que esse modelo "encaixa" melhor nos dados é o **ponto de mudança**.
3. Só declara que houve mudança se o encaixe for bom e a variação for grande em comparação com o ruído normal.
4. Lista os eventos (deploys, autoscaling, alertas) que aconteceram até 30 min antes ou depois desse ponto,
   dizendo se cada um veio **antes** ou **depois** da mudança. Um evento que veio depois não pode ser a causa.

In [7]:
def _ajustar_mudanca(tempos, valores):
    # Modelo: plano ate o instante tc; depois, reta partindo do nivel plano (continua em tc).
    n = len(valores)
    melhor = None
    for k in range(4, n - 3):
        a = sum(valores[:k]) / k
        tc = tempos[k]
        den = sum((tempos[i] - tc) ** 2 for i in range(k, n))
        b = sum((tempos[i] - tc) * (valores[i] - a) for i in range(k, n)) / den
        sse = sum((valores[i] - a) ** 2 for i in range(k))
        sse += sum((valores[i] - a - b * (tempos[i] - tc)) ** 2 for i in range(k, n))
        if melhor is None or sse < melhor[0]:
            melhor = (sse, k, a, b)
    return melhor


def analisar_serie_temporal(servico, metrica):
    erro = _validar_servico(servico)
    if erro:
        return erro
    if metrica not in METRICAS:
        return {"erro": f"Metrica desconhecida: {metrica}", "disponiveis": METRICAS}

    serie = ambiente["series"]
    valores = serie["dados"][servico][metrica]
    passo = serie["passo_min"]
    n = len(valores)
    tempos = [hhmm_para_min(serie["inicio"]) + i * passo for i in range(n)]

    sse, k, a, b = _ajustar_mudanca(tempos, valores)
    media = sum(valores) / n
    sse_plano = sum((v - media) ** 2 for v in valores)
    melhoria = 1 - sse / sse_plano if sse_plano > 0 else 0
    desvio_base = math.sqrt(sum((valores[i] - a) ** 2 for i in range(k)) / k)
    variacao_total = b * (tempos[-1] - tempos[k])
    houve_mudanca = melhoria >= 0.5 and abs(variacao_total) > 4 * max(desvio_base, 1e-9)

    passo_amostra = max(1, n // 12)
    indices = list(range(0, n, passo_amostra))
    if indices[-1] != n - 1:
        indices.append(n - 1)

    resultado = {
        "servico": servico,
        "metrica": metrica,
        "janela": f"{serie['inicio']}-{serie['fim']}",
        "resolucao_min": passo,
        "valor_atual": valores[-1],
        "amostras": [{"hora": min_para_hhmm(tempos[i]), "valor": valores[i]} for i in indices],
    }
    if not houve_mudanca:
        resultado["tendencia"] = "estavel (sem mudanca de tendencia relevante na janela)"
        resultado["baseline"] = round(media, 2)
        resultado["amostras"] = resultado.pop("amostras")   # amostras por ultimo (a tela corta o fim)
        return resultado

    tc = tempos[k]
    tolerancia = max(passo, 3)   # a estimativa do ponto de mudanca tem margem de erro de alguns minutos
    proximos = []
    for evento in ambiente["linha_do_tempo"]:
        diferenca = hhmm_para_min(evento["hora"]) - tc
        if abs(diferenca) <= 30:
            posicao = ("no mesmo instante (dentro da margem de erro)" if abs(diferenca) <= tolerancia
                       else ("antes da mudanca" if diferenca < 0 else "depois da mudanca"))
            proximos.append({**evento, "diferenca_min": diferenca, "posicao": posicao})
    proximos.sort(key=lambda e: abs(e["diferenca_min"]))

    resultado.update({
        "tendencia": "mudanca detectada",
        "baseline_antes_da_mudanca": round(a, 2),
        "mudanca_de_tendencia": min_para_hhmm(tc),
        "margem_de_erro_min": tolerancia,
        "inclinacao_apos_mudanca_por_hora": round(b * 60, 2),
        "variacao_pct_vs_baseline": round((valores[-1] - a) / a * 100, 1) if a else None,
        "eventos_proximos_da_mudanca": proximos,
    })
    resultado["amostras"] = resultado.pop("amostras")   # amostras por ultimo (a tela corta o fim)
    return resultado

### Teste da nova tool SEM usar a IA (Print: mostra que a ferramenta funciona sozinha)

In [8]:
carregar_cenario("B")
for servico, metrica in [("catalog-api", "memoria_pct"), ("checkout-api", "latencia_p95_ms"), ("checkout-api", "memoria_pct")]:
    r = analisar_serie_temporal(servico, metrica)
    r.pop("amostras")
    print(json.dumps(r, indent=2, ensure_ascii=False))
    print()

{
  "servico": "catalog-api",
  "metrica": "memoria_pct",
  "janela": "11:00-16:15",
  "resolucao_min": 5,
  "valor_atual": 91.53,
  "tendencia": "mudanca detectada",
  "baseline_antes_da_mudanca": 54.81,
  "mudanca_de_tendencia": "12:05",
  "margem_de_erro_min": 5,
  "inclinacao_apos_mudanca_por_hora": 8.77,
  "variacao_pct_vs_baseline": 67.0,
  "eventos_proximos_da_mudanca": [
    {
      "hora": "12:10",
      "tipo": "deploy",
      "servico": "catalog-api",
      "id": "dep-451",
      "diferenca_min": 5,
      "posicao": "no mesmo instante (dentro da margem de erro)"
    }
  ]
}

{
  "servico": "checkout-api",
  "metrica": "latencia_p95_ms",
  "janela": "11:00-16:15",
  "resolucao_min": 5,
  "valor_atual": 1068.99,
  "tendencia": "mudanca detectada",
  "baseline_antes_da_mudanca": 308.28,
  "mudanca_de_tendencia": "15:40",
  "margem_de_erro_min": 5,
  "inclinacao_apos_mudanca_por_hora": 1365.08,
  "variacao_pct_vs_baseline": 246.8,
  "eventos_proximos_da_mudanca": [
    {
      "

## Declaração das tools no formato Gemini

In [9]:
def decl_servico(nome, descricao):
    return {
        "name": nome,
        "description": descricao,
        "parameters": {
            "type": "OBJECT",
            "properties": {"servico": {"type": "STRING", "enum": SERVICOS, "description": "Nome do servico"}},
            "required": ["servico"],
        },
    }


DECLARACOES = [
    decl_servico("consultar_metricas", "Consulta CPU, memoria, latencia, erros e numero de instancias de um servico (valores atuais)."),
    decl_servico("buscar_logs", "Busca as linhas recentes de log de um servico. Use para investigar sintomas e erros."),
    decl_servico("consultar_deploys", "Lista deploys recentes e mudancas conhecidas de um servico."),
    decl_servico("consultar_configuracao", "Consulta a configuracao operacional atual de um servico."),
    decl_servico("consultar_custos", "Compara o custo atual por hora de um servico com seu baseline."),
    {
        "name": "consultar_eventos_recentes",
        "description": "Lista eventos cloud dos ultimos 90 minutos: deploys, autoscaling e alertas. Eventos mais antigos nao aparecem.",
    },
    {
        "name": "analisar_serie_temporal",
        "description": ("Analisa a evolucao de uma metrica ao longo do tempo. Informa QUANDO a metrica mudou de comportamento, "
                        "a taxa de crescimento depois disso e quais deploys/eventos ocorreram antes ou depois desse momento. "
                        "Use para descobrir o inicio real de um problema e testar hipoteses de causa."),
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "servico": {"type": "STRING", "enum": SERVICOS, "description": "Nome do servico"},
                "metrica": {"type": "STRING", "enum": METRICAS, "description": "Metrica a analisar"},
            },
            "required": ["servico", "metrica"],
        },
    },
    {
        "name": "solicitar_rollback",
        "description": "Solicita rollback de um deploy suspeito. Apenas abre pedido; exige aprovacao humana para executar.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "deploy_id": {"type": "STRING", "description": "ID exato de um deploy consultado"},
                "justificativa": {"type": "STRING", "description": "Evidencia objetiva que justifica o rollback"},
            },
            "required": ["deploy_id", "justificativa"],
        },
    },
]

FUNCOES_PERMITIDAS = {
    "consultar_metricas": consultar_metricas,
    "buscar_logs": buscar_logs,
    "consultar_deploys": consultar_deploys,
    "consultar_configuracao": consultar_configuracao,
    "consultar_custos": consultar_custos,
    "consultar_eventos_recentes": consultar_eventos_recentes,
    "analisar_serie_temporal": analisar_serie_temporal,
    "solicitar_rollback": solicitar_rollback,
}


def executar_ferramenta(nome, argumentos):
    if nome not in FUNCOES_PERMITIDAS:
        return {"erro": f"Ferramenta nao permitida: {nome}"}
    try:
        return FUNCOES_PERMITIDAS[nome](**argumentos)
    except Exception as exc:
        return {"erro": type(exc).__name__, "detalhe": str(exc)}


print(f"{len(DECLARACOES)} ferramentas declaradas.")

8 ferramentas declaradas.


## Instruções do agente (melhoradas)

Além da regra temporal, acrescentei três regras que vieram de **falhas observadas** na execução do cenário original:
(1) o agente pediu rollback sem ler o conteúdo do deploy; (2) escreveu o ID como `deploy-442` em vez de `dep-442`;
(3) afirmou "memory exhaustion" quando a memória estava em 84%.

In [10]:
INSTRUCOES_BASE = '''
Voce e o agente legista de incidentes Cloud da Loja Nebulosa.
Seu objetivo e investigar a causa raiz usando apenas evidencias retornadas pelas ferramentas.

Regras:
- Nao trate proximidade temporal como prova de causalidade.
- Investigue primeiro; nao chute valores ou eventos.
- Faca consultas curtas e seletivas.
- Diferencie sintoma, causa raiz e impacto financeiro.
- Antes de solicitar rollback, confira o conteudo do deploy com consultar_deploys.
- Cite IDs de deploy exatamente como retornados pelas ferramentas e nao afirme nada que as ferramentas nao mostraram.
- Se houver evidencia suficiente, solicite rollback com justificativa objetiva.
- Nunca diga que uma acao foi executada quando a ferramenta disser que aguarda aprovacao.
- Encerre com: diagnostico, evidencias, impacto, acao recomendada e nivel de confianca.
- Seja conciso.
'''.strip()

INSTRUCAO_TEMPORAL = '''
- Quando o problema for gradual, aparecer com atraso ou houver varios deploys candidatos, use analisar_serie_temporal para descobrir
  QUANDO cada metrica mudou. Um evento so pode ser causa se ocorreu ANTES do inicio do sintoma. Lembre que consultar_eventos_recentes
  mostra apenas os ultimos 90 minutos.
'''.strip()


def montar_instrucoes(usar_tool_temporal):
    return INSTRUCOES_BASE + ("\n" + INSTRUCAO_TEMPORAL if usar_tool_temporal else "")

## Chamada à API e laço do agente

Novidades em relação à v1.1: o agente registra todas as chamadas (`chamadas`) para a avaliação, pode rodar **sem** a tool temporal
(`usar_tool_temporal=False`, para comparar) e pode rodar em modo silencioso (`verboso=False`).

In [11]:
def chamar_gemini(corpo, tentativas=4):
    url = f"{BASE_URL}/models/{MODEL}:generateContent"
    for tentativa in range(1, tentativas + 1):
        req = urllib.request.Request(
            url,
            data=json.dumps(corpo).encode("utf-8"),
            method="POST",
            headers={"Content-Type": "application/json", "X-goog-api-key": os.environ["GEMINI_API_KEY"]},
        )
        try:
            with urllib.request.urlopen(req, timeout=90) as resposta:
                return json.loads(resposta.read())
        except urllib.error.HTTPError as erro:
            detalhe = erro.read().decode("utf-8", errors="replace")[:500]
            config = corpo.get("generationConfig", {})
            if erro.code == 400 and "thinking" in detalhe.lower() and "thinkingConfig" in config:
                print("  (modelo nao aceita thinkingConfig; repetindo sem ele)")
                del config["thinkingConfig"]
                continue
            if erro.code in (429, 500, 503) and tentativa < tentativas:
                espera = 15 * tentativa
                print(f"  (API respondeu {erro.code}; aguardando {espera}s para tentar de novo...)")
                time.sleep(espera)
                continue
            raise RuntimeError(f"Erro {erro.code} da API Gemini: {detalhe}") from None


def estimar_custo(input_tokens, output_tokens):
    return input_tokens * PRECO_INPUT_1M / 1_000_000 + output_tokens * PRECO_OUTPUT_1M / 1_000_000


def investigar_incidente(incidente=None, usar_tool_temporal=True, verboso=True):
    def falar(*args):
        if verboso:
            print(*args)

    incidente = incidente or ambiente["incidente"]
    declaracoes = [d for d in DECLARACOES if usar_tool_temporal or d["name"] != "analisar_serie_temporal"]
    historico = [{"role": "user", "parts": [{"text": incidente}]}]
    total_input = total_output = 0
    chamadas = []

    def resumo(resposta_final, interrompido, rodada):
        return {
            "resposta": resposta_final,
            "interrompido": interrompido,
            "rodadas": rodada,
            "input_tokens": total_input,
            "output_tokens": total_output,
            "custo_equivalente_usd": estimar_custo(total_input, total_output),
            "ferramentas": [nome for nome, _ in chamadas],
            "chamadas": chamadas,
        }

    for rodada in range(1, MAX_RODADAS + 1):
        corpo = {
            "systemInstruction": {"parts": [{"text": montar_instrucoes(usar_tool_temporal)}]},
            "contents": historico,
            "tools": [{"functionDeclarations": declaracoes}],
            "toolConfig": {"functionCallingConfig": {"mode": "AUTO"}},
            "generationConfig": {"maxOutputTokens": MAX_OUTPUT_TOKENS,
                                 "thinkingConfig": {"thinkingLevel": THINKING_LEVEL}},
        }
        resposta = chamar_gemini(corpo)
        uso = resposta.get("usageMetadata", {})
        total_input += uso.get("promptTokenCount", 0)
        total_output += uso.get("candidatesTokenCount", 0) + uso.get("thoughtsTokenCount", 0)
        custo = estimar_custo(total_input, total_output)
        falar(f"\n--- Rodada {rodada} | custo equivalente acumulado: US$ {custo:.6f} (real: US$ 0, plano gratuito) ---")

        candidatos = resposta.get("candidates") or []
        if not candidatos:
            falar("A API nao devolveu candidatos:", json.dumps(resposta.get("promptFeedback", {}), ensure_ascii=False))
            return resumo(None, True, rodada)

        conteudo = candidatos[0].get("content", {"role": "model", "parts": []})
        partes = conteudo.get("parts", [])
        historico.append(conteudo)  # preserva a thoughtSignature
        pedidos = [p["functionCall"] for p in partes if "functionCall" in p]

        if not pedidos:
            texto = "".join(p.get("text", "") for p in partes if not p.get("thought"))
            if candidatos[0].get("finishReason") == "MAX_TOKENS":
                falar("AVISO: resposta cortada por MAX_OUTPUT_TOKENS; aumente o limite.")
            falar("\nRELATORIO FINAL\n")
            falar(texto)
            return resumo(texto, False, rodada)

        retornos = []
        for pedido in pedidos:
            nome, argumentos = pedido["name"], pedido.get("args", {})
            falar(f"Agente pediu: {nome}({argumentos})")
            resultado = executar_ferramenta(nome, argumentos)
            texto_obs = json.dumps(resultado, ensure_ascii=False)
            falar("Observacao:", texto_obs if len(texto_obs) < 700 else texto_obs[:700] + " ...[cortado na tela]")
            chamadas.append((nome, argumentos))
            retorno = {"name": nome, "response": {"result": resultado}}
            if "id" in pedido:
                retorno["id"] = pedido["id"]
            retornos.append({"functionResponse": retorno})
        historico.append({"role": "user", "parts": retornos})

        if custo >= LIMITE_ESTIMADO_USD:
            falar(f"\nLimite didatico de US$ {LIMITE_ESTIMADO_USD:.2f} (equivalente) atingido.")
            return resumo(None, True, rodada)

    falar("\nNumero maximo de rodadas atingido. O agente foi interrompido pelo software.")
    return resumo(None, True, MAX_RODADAS)

## Execução 1: cenário B (o problema novo), com a tool temporal

In [12]:
carregar_cenario("B")
resultado_B = investigar_incidente()


--- Rodada 1 | custo equivalente acumulado: US$ 0.000467 (real: US$ 0, plano gratuito) ---
Agente pediu: consultar_metricas({'servico': 'checkout-api'})
Observacao: {"servico": "checkout-api", "cpu_pct": 47.98, "memoria_pct": 58.88, "latencia_p95_ms": 1068.99, "taxa_erro_pct": 4.86, "instancias": 8}
Agente pediu: analisar_serie_temporal({'metrica': 'latencia_p95_ms', 'servico': 'checkout-api'})
Observacao: {"servico": "checkout-api", "metrica": "latencia_p95_ms", "janela": "11:00-16:15", "resolucao_min": 5, "valor_atual": 1068.99, "tendencia": "mudanca detectada", "baseline_antes_da_mudanca": 308.28, "mudanca_de_tendencia": "15:40", "margem_de_erro_min": 5, "inclinacao_apos_mudanca_por_hora": 1365.08, "variacao_pct_vs_baseline": 246.8, "eventos_proximos_da_mudanca": [{"hora": "15:52", "tipo": "deploy", "servico": "checkout-api", "id": "dep-455", "diferenca_min": 12, "posicao": "depois da mudanca"}, {"hora": "16:05", "tipo": "autoscaling", "servico": "checkout-api", "detalhe": "5 -> 8 

## Execução 2: cenário A (o original), para provar que a tool nova não quebrou o que já funcionava

In [13]:
carregar_cenario("A")
resultado_A = investigar_incidente()

  (API respondeu 503; aguardando 15s para tentar de novo...)

--- Rodada 1 | custo equivalente acumulado: US$ 0.000475 (real: US$ 0, plano gratuito) ---
Agente pediu: consultar_metricas({'servico': 'api-gateway'})
Observacao: {"servico": "api-gateway", "cpu_pct": 38, "memoria_pct": 61, "latencia_p95_ms": 6200, "taxa_erro_pct": 18.4, "instancias": 3}
Agente pediu: consultar_custos({'servico': 'api-gateway'})
Observacao: {"servico": "api-gateway", "baseline_usd_h": 0.22, "atual_usd_h": 0.24, "multiplicador": 1.1}
Agente pediu: analisar_serie_temporal({'metrica': 'latencia_p95_ms', 'servico': 'api-gateway'})
Observacao: {"servico": "api-gateway", "metrica": "latencia_p95_ms", "janela": "13:00-14:08", "resolucao_min": 1, "valor_atual": 6200, "tendencia": "mudanca detectada", "baseline_antes_da_mudanca": 139.51, "mudanca_de_tendencia": "14:02", "margem_de_erro_min": 3, "inclinacao_apos_mudanca_por_hora": 76285.75, "variacao_pct_vs_baseline": 4344.3, "eventos_proximos_da_mudanca": [{"hora": 

## Avaliação automática

Cada caso é executado `REPETICOES` vezes. O critério de acerto é objetivo e verificado pelo código, não por opinião:
- o agente **pediu rollback do deploy correto** (A: `dep-442`; B: `dep-451`);
- e **não pediu rollback do deploy errado** (B: `dep-455`, a pista falsa);
- e terminou com um relatório (não foi interrompido).

Também registramos se usou a tool temporal, quantas rodadas gastou, tokens e custo equivalente.
Os resultados variam de uma execução para outra (o modelo não é 100% determinístico), por isso repetimos.

In [14]:
REPETICOES = 3

CASOS = [
    # nome, cenario, usar_tool_temporal, rollback esperado, rollbacks proibidos
    ("A original, com tool temporal", "A", True, "dep-442", []),
    ("B temporal, SEM tool temporal", "B", False, "dep-451", ["dep-455"]),
    ("B temporal, COM tool temporal", "B", True, "dep-451", ["dep-455"]),
]


def rodar_um_caso(cenario, usar_tool, esperado, proibidos):
    carregar_cenario(cenario)
    try:
        r = investigar_incidente(usar_tool_temporal=usar_tool, verboso=False)
    except Exception as exc:
        return {"acertou": False, "erro": str(exc)[:120], "rodadas": 0, "tokens": 0, "custo": 0.0, "usou_temporal": False}
    rollbacks = [args.get("deploy_id") for nome, args in r["chamadas"] if nome == "solicitar_rollback"]
    acertou = (not r["interrompido"]) and esperado in rollbacks and not any(p in rollbacks for p in proibidos)
    return {
        "acertou": acertou,
        "rollbacks": rollbacks,
        "usou_temporal": "analisar_serie_temporal" in r["ferramentas"],
        "rodadas": r["rodadas"],
        "tokens": r["input_tokens"] + r["output_tokens"],
        "custo": r["custo_equivalente_usd"],
        "ferramentas": r["ferramentas"],
    }


def avaliar_tudo(repeticoes=REPETICOES):
    tabela = []
    for nome, cenario, usar_tool, esperado, proibidos in CASOS:
        rodadas = []
        for i in range(repeticoes):
            print(f"Executando: {nome} (tentativa {i + 1}/{repeticoes})...")
            rodadas.append(rodar_um_caso(cenario, usar_tool, esperado, proibidos))
            time.sleep(3)
        ok = [x for x in rodadas if "erro" not in x]
        tabela.append({
            "caso": nome,
            "acertos": f"{sum(x['acertou'] for x in rodadas)}/{repeticoes}",
            "usou_temporal": f"{sum(x['usou_temporal'] for x in rodadas)}/{repeticoes}",
            "rodadas_media": round(sum(x["rodadas"] for x in ok) / max(1, len(ok)), 1),
            "tokens_media": round(sum(x["tokens"] for x in ok) / max(1, len(ok))),
            "custo_eq_media_usd": round(sum(x["custo"] for x in ok) / max(1, len(ok)), 5),
            "rollbacks": [x.get("rollbacks") for x in rodadas],
            "erros": [x["erro"] for x in rodadas if "erro" in x],
        })
    print("\n" + "=" * 100)
    print(f"{'Caso':34} {'Acertos':8} {'Usou temp.':11} {'Rodadas':8} {'Tokens':8} {'Custo eq. (US$)':16}")
    for t in tabela:
        print(f"{t['caso']:34} {t['acertos']:8} {t['usou_temporal']:11} {t['rodadas_media']:<8} {t['tokens_media']:<8} {t['custo_eq_media_usd']:<16}")
    print("=" * 100)
    for t in tabela:
        print(t["caso"], "-> rollbacks pedidos em cada tentativa:", t["rollbacks"], t["erros"] or "")
    return tabela

In [15]:
tabela = avaliar_tudo()

Executando: A original, com tool temporal (tentativa 1/3)...
  (API respondeu 503; aguardando 15s para tentar de novo...)
  (API respondeu 503; aguardando 30s para tentar de novo...)
  (API respondeu 503; aguardando 45s para tentar de novo...)
Executando: A original, com tool temporal (tentativa 2/3)...
Executando: A original, com tool temporal (tentativa 3/3)...
  (API respondeu 503; aguardando 15s para tentar de novo...)
  (API respondeu 503; aguardando 15s para tentar de novo...)
Executando: B temporal, SEM tool temporal (tentativa 1/3)...
  (API respondeu 503; aguardando 15s para tentar de novo...)
  (API respondeu 503; aguardando 15s para tentar de novo...)
Executando: B temporal, SEM tool temporal (tentativa 2/3)...
  (API respondeu 503; aguardando 15s para tentar de novo...)
Executando: B temporal, SEM tool temporal (tentativa 3/3)...
  (API respondeu 503; aguardando 15s para tentar de novo...)
Executando: B temporal, COM tool temporal (tentativa 1/3)...
  (API respondeu 503; ag

## Roteiro do professor: situação

1. Tool de análise temporal: **feito** (`analisar_serie_temporal`)
   1.1. Mockup de métricas com problema novo que exige investigação temporal: **feito** (cenário B)
2. Testar e avaliar: **feito** (execuções 1 e 2 e tabela de avaliação)
3. Portar para nova LLM: **feito** (OpenAI -> Gemini)